In [ ]:
%maven tools.aqua:z3-turnkey:4.14.1
import com.microsoft.z3.*;
import java.util.stream.*;

The function `range(n)` returns the list `[0, ..., n-1]`.

In [ ]:
List<Integer> range(int n) {
    return IntStream.range(0, n).boxed().toList();
}

In [ ]:
range(5)

# Sudoku

The sudoku we want to solve is shown here:
    <img src="sudoku.png" width="40%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The function `createPuzzle` returns a representation of this puzzle as a two-dimensional array.  The number `0` denotes a cell that is empty.

In [ ]:
int[][] createPuzzle() {
    return new int[][] {
        { 0, 3, 9, 0, 0, 0, 0, 0, 7 },
        { 0, 0, 0, 7, 0, 0, 4, 9, 2 },
        { 0, 0, 0, 0, 6, 5, 0, 8, 3 },
        { 0, 0, 0, 6, 0, 3, 2, 7, 0 },
        { 0, 0, 0, 0, 4, 0, 8, 0, 0 },
        { 5, 6, 0, 0, 0, 0, 0, 0, 0 },
        { 0, 0, 5, 2, 0, 9, 0, 0, 1 },
        { 0, 2, 1, 0, 0, 0, 0, 4, 0 },
        { 7, 0, 0, 0, 0, 0, 5, 0, 0 }
    };
}

We are going to solve this puzzle with the help of the constraint solver `Z3`.  All `Z3` objects are created by a *context*, which we call `Z3`.

In [ ]:
Context Z3 = new Context();

The function `constraintsFromPuzzle` takes one argument:
* `Variables` is a matrix of `Z3` variables.

   For `row`$\in \{0,8\}$ and `col`$\in \{0,8\}$ the variable `Variables[row][col]` specifies the number that is placed in the specified row and column.

It returns a list of constraints specifying that the variables corresponding to numbers that are already set in the given Sudoku take the specified values.

In [ ]:
List<BoolExpr> constraintsFromPuzzle(IntExpr[][] Variables) {
    var Puzzle = createPuzzle();
    return List.of();  // your code here
}

In [ ]:
IntExpr[][] Variables = range(9).stream()
    .map(row -> range(9).stream().map(col -> Z3.mkIntConst("V" + (row + 1) + (col + 1))).toArray(IntExpr[]::new))
    .toArray(IntExpr[][]::new);

The function `allConstraints` returns a list of constraints that encode the given sudoku.  The method `Z3.mkDistinct` takes a variable number of arguments and specifies that all of them are different.

In [ ]:
List<BoolExpr> allConstraints(IntExpr[][] Variables) {
    var constraints    = constraintsFromPuzzle(Variables);
    var rowConstraints = range(9).stream().map(row -> Z3.mkDistinct(Variables[row])).toList();
    List<BoolExpr> colConstraints    = List.of();  // your code here
    List<BoolExpr> squareConstraints = List.of();  // your code here
    List<BoolExpr> boundConstraints  = List.of();  // your code here
    List<BoolExpr> result = new ArrayList<>(constraints);
    result.addAll(rowConstraints);
    result.addAll(colConstraints);
    result.addAll(squareConstraints);
    result.addAll(boundConstraints);
    return result;
}

I have got 217 constraints.

In [ ]:
allConstraints(Variables).size()

The function `solve()` computes a solution to the given problem and returns this solution.  The solution is a map that maps the names of the variables to their values.  If there is no solution, `null` is returned.

In [ ]:
Map<String, Integer> solve() {
    IntExpr[][] Variables = range(9).stream()
        .map(row -> range(9).stream().map(col -> Z3.mkIntConst("V" + (row + 1) + (col + 1))).toArray(IntExpr[]::new))
        .toArray(IntExpr[][]::new);
    Solver S = Z3.mkSolver();
    S.add(allConstraints(Variables).toArray(BoolExpr[]::new));
    var result = S.check();
    if (result == Status.SATISFIABLE) {
        Model M = S.getModel();
        Map<String, Integer> Solution = new LinkedHashMap<>();
        for (int row : range(9)) {
            for (int col : range(9)) {
                String varName = "V" + (row + 1) + (col + 1);
                Solution.put(varName, Integer.parseInt(M.eval(Variables[row][col], true).toString()));
            }
        }
        return Solution;
    } else if (result == Status.UNSATISFIABLE) {
        System.out.println("The problem is not solvable.");
    } else {
        System.out.println("Z3 cannot determine whether the problem is solvable.");
    }
    return null;
}

In [ ]:
long start   = System.currentTimeMillis();
var Solution = solve();
System.out.println("Solver Duration: " + (System.currentTimeMillis() - start) + " ms");
Solution

## Graphical Representation

The function `showSolution` displays the given solution on a 9x9 grid, rendering it as an SVG image.  The digits given in the puzzle are shown in bold.

In [ ]:
void showSolution(Map<String, Integer> Solution, String width) {
    if (Solution == null) {
        System.out.println("No solution to display.");
        return;
    }
    var Sudoku    = createPuzzle();
    int cellSize  = 50;
    int totalSize = 9 * cellSize;
    var html = new StringBuilder();
    html.append("<svg width=\"%s\" viewBox=\"-5 -5 %d %d\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: sans-serif;\">"
                .formatted(width, totalSize + 10, totalSize + 10));
    for (int r : range(9)) {
        for (int c : range(9)) {
            int     x       = c * cellSize;
            int     y       = r * cellSize;
            boolean isGiven = Sudoku[r][c] != 0;
            int     val     = isGiven ? Sudoku[r][c] : Solution.get("V" + (r + 1) + (c + 1));
            String  bg      = isGiven ? "#f0f0f0" : "#ffffff";
            String  color   = isGiven ? "#000000" : "#1a73e8";
            String  weight  = isGiven ? "bold" : "normal";
            html.append("<rect x=\"%d\" y=\"%d\" width=\"%d\" height=\"%d\" fill=\"%s\" stroke=\"#cccccc\" stroke-width=\"1\" />"
                        .formatted(x, y, cellSize, cellSize, bg));
            html.append("<text x=\"%d\" y=\"%d\" text-anchor=\"middle\" font-size=\"22\" fill=\"%s\" font-weight=\"%s\">%d</text>"
                        .formatted(x + cellSize / 2, y + cellSize / 2 + 7, color, weight, val));
        }
    }
    for (int i : List.of(0, 3, 6, 9)) {
        int pos = i * cellSize;
        html.append("<line x1=\"0\" y1=\"%d\" x2=\"%d\" y2=\"%d\" stroke=\"#333333\" stroke-width=\"3\" stroke-linecap=\"square\" />"
                    .formatted(pos, totalSize, pos));
        html.append("<line x1=\"%d\" y1=\"0\" x2=\"%d\" y2=\"%d\" stroke=\"#333333\" stroke-width=\"3\" stroke-linecap=\"square\" />"
                    .formatted(pos, pos, totalSize));
    }
    html.append("</svg>");
    display(html.toString(), "text/html");
}

void showSolution(Map<String, Integer> Solution) {
    showSolution(Solution, "50%");
}

In [ ]:
showSolution(Solution);